In [1]:
import logging

import numpy as np
import pandas as pd
import akshare as ak

import warnings
warnings.filterwarnings("ignore", category=UserWarning)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

In [2]:
df = ak.stock_zh_a_tick_tx_js(symbol='sz002487')

In [14]:
df.to_excel('1.xlsx')

In [3]:
df.head()

,成交时间,成交价格,价格变动,成交量,成交金额,性质
0,09:25:00,70.36,0.46,1315,9252340,买盘
1,09:30:00,70.50,0.14,834,5871373,买盘
2,09:30:03,70.48,-0.02,374,2638263,卖盘
3,09:30:06,70.42,-0.06,652,4590780,卖盘
4,09:30:09,70.37,-0.05,264,1858127,卖盘


In [22]:
# ===== 1️⃣ 转时间格式 =====
df["成交时间"] = pd.to_datetime(df["成交时间"])

# ===== 2️⃣ 提取分钟 =====
df["分钟"] = df["成交时间"].dt.floor("min")

# ===== 3️⃣ 标记买卖 =====
df["买量"] = df.apply(lambda x: x["成交量"] if x["性质"] == "买盘" or (x["性质"]=="中性盘" and x["价格变动"]>0) else 0, axis=1)

df["卖量"] = df.apply(lambda x: x["成交量"] if x["性质"] == "卖盘" or (x["性质"]=="中性盘" and x["价格变动"]<0) else 0, axis=1)

# ===== 4️⃣ 按分钟聚合 =====
minute_df = df.groupby("分钟").agg({
    "买量": "sum",
    "卖量": "sum",
    "成交量": "sum"
}).reset_index()

# ===== 5️⃣ 计算买卖比 =====
minute_df["买卖比"] = minute_df["买量"] / (minute_df["卖量"] + 1e-6)
minute_df["买卖差"] = minute_df["买量"] - minute_df["卖量"]
minute_df["买卖比"] = minute_df["买卖比"].round(2)
minute_df["买卖比"] = minute_df["买卖比"].map(lambda x: f"{x:.2f}")
print(minute_df)

                     分钟    买量    卖量   成交量            买卖比   买卖差
0   2026-03-22 09:25:00  1315     0  1315  1315000000.00  1315
1   2026-03-22 09:30:00  3166  3218  6569           0.98   -52
2   2026-03-22 09:31:00  2424   774  3198           3.13  1650
3   2026-03-22 09:32:00  1556  1435  2991           1.08   121
4   2026-03-22 09:33:00  1069  1728  2797           0.62  -659
..                  ...   ...   ...   ...            ...   ...
236 2026-03-22 14:54:00   388  1050  1438           0.37  -662
237 2026-03-22 14:55:00  1304   916  2220           1.42   388
238 2026-03-22 14:56:00  1184  1671  2855           0.71  -487
239 2026-03-22 14:57:00    58     0    58    58000000.00    58
240 2026-03-22 15:00:00     0  3583  3583           0.00 -3583

[241 rows x 6 columns]


In [23]:
minute_df["order_flow"] = (
    minute_df["买量"] - minute_df["卖量"]
) / (minute_df["成交量"] + 1e-6)

In [24]:
minute_df["买占比"] = minute_df["买量"] / (minute_df["成交量"] + 1e-6)

In [25]:
minute_df.head(20)

,分钟,买量,卖量,成交量,买卖比,买卖差,order_flow,买占比
0,2026-03-22 09:25:00,1315,0,1315,1315000000.00,1315,1.000000,1.000000
1,2026-03-22 09:30:00,3166,3218,6569,0.98,-52,-0.007916,0.481961
2,2026-03-22 09:31:00,2424,774,3198,3.13,1650,0.515947,0.757974
3,2026-03-22 09:32:00,1556,1435,2991,1.08,121,0.040455,0.520227
4,2026-03-22 09:33:00,1069,1728,2797,0.62,-659,-0.235610,0.382195
5,2026-03-22 09:34:00,323,2004,2327,0.16,-1681,-0.722389,0.138805
6,2026-03-22 09:35:00,1579,1169,2774,1.35,410,0.147801,0.569214
7,2026-03-22 09:36:00,2207,654,2861,3.37,1553,0.542817,0.771409
8,2026-03-22 09:37:00,5116,1384,6500,3.70,3732,0.574154,0.787077
9,2026-03-22 09:38:00,2344,2155,4499,1.09,189,0.042009,0.521005


In [28]:
df["累计成交额"] = df["成交金额"].cumsum()
df["累计成交量"] = df["成交量"].cumsum()*100
df["VWAP"] = df["累计成交额"] / df["累计成交量"]

In [30]:
df.head(20)

,成交时间,成交价格,价格变动,成交量,成交金额,性质,分钟,买量,卖量,累计成交额,累计成交量,VWAP
0,2026-03-22 09:25:00,70.36,0.46,1315,9252340,买盘,2026-03-22 09:25:00,1315,0,9252340,131500,70.360000
1,2026-03-22 09:30:00,70.50,0.14,834,5871373,买盘,2026-03-22 09:30:00,834,0,15123713,214900,70.375584
2,2026-03-22 09:30:03,70.48,-0.02,374,2638263,卖盘,2026-03-22 09:30:00,0,374,17761976,252300,70.400222
3,2026-03-22 09:30:06,70.42,-0.06,652,4590780,卖盘,2026-03-22 09:30:00,0,652,22352756,317500,70.402381
4,2026-03-22 09:30:09,70.37,-0.05,264,1858127,卖盘,2026-03-22 09:30:00,0,264,24210883,343900,70.400939
5,2026-03-22 09:30:12,70.36,-0.01,518,3645070,卖盘,2026-03-22 09:30:00,0,518,27855953,395700,70.396646
6,2026-03-22 09:30:15,70.32,-0.04,185,1301441,卖盘,2026-03-22 09:30:00,0,185,29157394,414200,70.394481
7,2026-03-22 09:30:18,70.25,-0.07,80,562356,卖盘,2026-03-22 09:30:00,0,80,29719750,422200,70.392586
8,2026-03-22 09:30:21,70.13,-0.12,65,456086,卖盘,2026-03-22 09:30:00,0,65,30175836,428700,70.389167
9,2026-03-22 09:30:24,70.00,-0.13,151,1057270,卖盘,2026-03-22 09:30:00,0,151,31233106,443800,70.376534


In [31]:
df["VWAP斜率"] = df["VWAP"].diff()
df["均价抬高"] = df["VWAP斜率"] > 0

In [32]:
df.head(20)

,成交时间,成交价格,价格变动,成交量,成交金额,性质,分钟,买量,卖量,累计成交额,累计成交量,VWAP,VWAP斜率,均价抬高
0,2026-03-22 09:25:00,70.36,0.46,1315,9252340,买盘,2026-03-22 09:25:00,1315,0,9252340,131500,70.360000,NaN,False
1,2026-03-22 09:30:00,70.50,0.14,834,5871373,买盘,2026-03-22 09:30:00,834,0,15123713,214900,70.375584,0.015584,True
2,2026-03-22 09:30:03,70.48,-0.02,374,2638263,卖盘,2026-03-22 09:30:00,0,374,17761976,252300,70.400222,0.024638,True
3,2026-03-22 09:30:06,70.42,-0.06,652,4590780,卖盘,2026-03-22 09:30:00,0,652,22352756,317500,70.402381,0.002159,True
4,2026-03-22 09:30:09,70.37,-0.05,264,1858127,卖盘,2026-03-22 09:30:00,0,264,24210883,343900,70.400939,-0.001442,False
5,2026-03-22 09:30:12,70.36,-0.01,518,3645070,卖盘,2026-03-22 09:30:00,0,518,27855953,395700,70.396646,-0.004293,False
6,2026-03-22 09:30:15,70.32,-0.04,185,1301441,卖盘,2026-03-22 09:30:00,0,185,29157394,414200,70.394481,-0.002166,False
7,2026-03-22 09:30:18,70.25,-0.07,80,562356,卖盘,2026-03-22 09:30:00,0,80,29719750,422200,70.392586,-0.001894,False
8,2026-03-22 09:30:21,70.13,-0.12,65,456086,卖盘,2026-03-22 09:30:00,0,65,30175836,428700,70.389167,-0.003419,False
9,2026-03-22 09:30:24,70.00,-0.13,151,1057270,卖盘,2026-03-22 09:30:00,0,151,31233106,443800,70.376534,-0.012633,False


In [84]:

import baostock as bs
import pandas as pd

#### 登陆系统 ####
lg = bs.login()
# 显示登陆返回信息
print('login respond error_code:'+lg.error_code)
print('login respond  error_msg:'+lg.error_msg)

#### 获取沪深A股历史K线数据 ####
# 详细指标参数，参见“历史行情指标参数”章节；“分钟线”参数与“日线”参数不同。“分钟线”不包含指数。
# 分钟线指标：date,time,code,open,high,low,close,volume,amount,adjustflag
# 周月线指标：date,code,open,high,low,close,volume,amount,adjustflag,turn,pctChg
rs = bs.query_history_k_data_plus("sh.600468",
    "date,time,code,open,high,low,close,volume,amount,adjustflag",
    start_date='2025-01-01', end_date='2026-03-22',
    frequency="5", adjustflag="3")
print('query_history_k_data_plus respond error_code:'+rs.error_code)
print('query_history_k_data_plus respond  error_msg:'+rs.error_msg)

#### 打印结果集 ####
data_list = []
while (rs.error_code == '0') & rs.next():
    # 获取一条记录，将记录合并在一起
    data_list.append(rs.get_row_data())
result = pd.DataFrame(data_list, columns=rs.fields)

#### 结果集输出到csv文件 ####   
# result.to_csv("D:\\history_A_stock_k_data.csv", index=False)
print(result)

#### 登出系统 ####
bs.logout()


login success!
login respond error_code:0
login respond  error_msg:success
query_history_k_data_plus respond error_code:0
query_history_k_data_plus respond  error_msg:success
             date               time       code    open    high     low  \
0      2025-01-02  20250102093500000  sh.600468  4.6900  4.7100  4.6200   
1      2025-01-02  20250102094000000  sh.600468  4.6300  4.6900  4.6300   
2      2025-01-02  20250102094500000  sh.600468  4.6800  4.7000  4.6600   
3      2025-01-02  20250102095000000  sh.600468  4.7000  4.7100  4.6800   
4      2025-01-02  20250102095500000  sh.600468  4.6800  4.7100  4.6800   
...           ...                ...        ...     ...     ...     ...   
14011  2026-03-20  20260320144000000  sh.600468  8.7200  8.7200  8.6800   
14012  2026-03-20  20260320144500000  sh.600468  8.7000  8.7200  8.6900   
14013  2026-03-20  20260320145000000  sh.600468  8.6900  8.7000  8.6700   
14014  2026-03-20  20260320145500000  sh.600468  8.6900  8.7000  8.6600   


In [88]:
result.head()

,date,time,code,open,high,low,close,volume,amount,adjustflag,hour,minute,pre_close
0,2025-01-02,2025-01-02 09:35:00,sh.600468,4.6900,4.7100,4.6200,4.6300,1360500,6329752.0000,3,9,35,None
1,2025-01-02,2025-01-02 09:40:00,sh.600468,4.6300,4.6900,4.6300,4.6700,566700,2639773.0000,3,9,40,4.6300
2,2025-01-02,2025-01-02 09:45:00,sh.600468,4.6800,4.7000,4.6600,4.7000,504000,2359430.0000,3,9,45,4.6700
3,2025-01-02,2025-01-02 09:50:00,sh.600468,4.7000,4.7100,4.6800,4.6800,389700,1829253.0000,3,9,50,4.7000
4,2025-01-02,2025-01-02 09:55:00,sh.600468,4.6800,4.7100,4.6800,4.7000,421100,1978966.0000,3,9,55,4.6800


In [121]:
df=result
# df['time']=df['time'].map(lambda x:x[:12])
df['time']=pd.to_datetime(df['time'],format="%Y%m%d%H%M%S%f")
df['hour']=df['time'].dt.hour
df['minute']=df['time'].dt.minute

df['pre_close']=df['close'].shift(1)
df=df.loc[df['date']!='2025-01-02']

res_df = []
for date, v in df.groupby('date'):
    # 1. 计算当天最高价 (这是标量，没问题)
    today_high = v['high'].max()
    # 2. 提取特定时间的数据，并强制转换为标量 (使用 .iloc[0])
    # 添加简单的错误处理，防止某天缺少开盘或收盘数据
    
    # 提取开盘 (09:35)
    open_series = v.loc[(v['hour']==9) & (v['minute']==35), 'open']
    if open_series.empty:
        continue # 或者设为 np.nan
    today_open = open_series.iloc[0]
    
    # 提取收盘 (15:00)
    close_series = v.loc[(v['hour']==15) & (v['minute']==0), 'close']
    if close_series.empty:
        continue
    today_close = close_series.iloc[0]
    
    # 提取昨收 (09:35 的 pre_close)
    yes_close_series = v.loc[(v['hour']==9) & (v['minute']==35), 'pre_close']
    if yes_close_series.empty:
        continue
    yes_close = yes_close_series.iloc[0]
    
    # 3. 确保是浮点数进行计算
    today_open = float(today_open)
    today_close = float(today_close)
    yes_close = float(yes_close)
    
    # 4. 计算涨幅 (此时都是数字，不会报错)
    # 防止除以0
    if yes_close != 0:
        zf = round((today_close - yes_close) / yes_close, 2)
    else:
        zf = 0.0
        
    if today_open != 0:
        sjzf = round((today_close - today_open) / today_open, 2)
    else:
        sjzf = 0.0
    
    # 5. 赋值给新列
    # 因为今天是标量 (scalar)，Pandas 会自动广播到该组的所有行
    v['today_high'] = today_high   # 修正了变量名拼写 hight -> high
    v['today_open'] = today_open
    v['today_close'] = today_close
    v['yes_close'] = yes_close
    v['zf'] = zf
    v['sjzf'] = sjzf
    v=v.iloc[0:3]
    res_df.append(v)

# 合并数据
if res_df:
    dfs = pd.concat(res_df, ignore_index=False) # 保持原有索引或重置
    print("处理完成，前5行数据：")
    print(dfs.head())
else:
    print("未生成任何数据，请检查时间筛选条件是否匹配。")

def estimate_bar_buy_ratio(row):
    high = row['high']
    low = row['low']
    close = row['close']
    
    # 处理可能的缺失值 (NaN)
    if pd.isna(high) or pd.isna(low) or pd.isna(close):
        return np.nan
    range_val = high - low
    # print(range_val)
    if range_val == 0:
        return 0.5 # 无波动视为中性
    
    ratio = (close - low) / range_val
    return round(ratio,2)

dfs['high']=dfs['high'].astype(float)
dfs['low']=dfs['low'].astype(float)
dfs['close']=dfs['close'].astype(float)
dfs['buy_ratio']=dfs.apply(estimate_bar_buy_ratio,axis=1)


dfs['volume']=dfs['volume'].astype(int)
dfs['est_buy_vol'] = dfs['volume'] * dfs['buy_ratio']
dfs['est_sell_vol'] = dfs['volume'] * (1 - dfs['buy_ratio'])


min15_df=[]
for k,v in dfs.groupby('date'):
    buy_volume=v['est_buy_vol'].sum()
    sell_volume=v['est_sell_vol'].sum()
    # if sell_volume==0:
    #     buy_ratio=-1
    # else:
    #     buy_ratio=buy_volume/sell_volume

    buy_ratio=(buy_volume+1e-8)/(sell_volume+1e-8)
    if buy_ratio>50:
        buy_ratio=50
    all_volume=buy_volume+sell_volume
    zb=round(all_volume/1087735321,4)
    v.reset_index(drop=True,inplace=True)
    zf=v.loc[0,'zf']
    sjzf=v.loc[0,'sjzf']
    dt={
        "buy_volume":buy_volume,
        "sell_volume":sell_volume,
        "buy_ratio":buy_ratio,
        "all_volume":all_volume,
        "zb":zb,
        "zf":zf,
        "sjzf":sjzf,
        "date":k
    }
    min15_df.append(dt)
min15_df=pd.DataFrame(min15_df)
min15_df['next_day_zf']=min15_df['zf'].shift(-1)

处理完成，前5行数据：
          date                time       code    open    high     low   close  \
48  2025-01-03 2025-01-03 09:35:00  sh.600468  4.5700  4.6100  4.5000  4.5100   
49  2025-01-03 2025-01-03 09:40:00  sh.600468  4.5100  4.5400  4.4800  4.5400   
50  2025-01-03 2025-01-03 09:45:00  sh.600468  4.5200  4.5600  4.5200  4.5500   
96  2025-01-06 2025-01-06 09:35:00  sh.600468  4.3600  4.3700  4.2600  4.3100   
97  2025-01-06 2025-01-06 09:40:00  sh.600468  4.3100  4.3700  4.3000  4.3500   

     volume        amount adjustflag  hour  minute pre_close today_high  \
48  1421000  6478995.0000          3     9      35    4.6000     4.6100   
49  1066300  4805089.0000          3     9      40    4.5100     4.6100   
50   459500  2088137.0000          3     9      45    4.5400     4.6100   
96  1840000  7906074.0000          3     9      35    4.3900     4.4800   
97   773200  3357184.0000          3     9      40    4.3100     4.4800   

    today_open  today_close  yes_close    zf  sjzf

In [123]:
min_buy_ratio=min15_df['buy_ratio'].min()
max_buy_ratio=min15_df['buy_ratio'].max()

if max_buy_ratio - min_buy_ratio != 0:
    min15_df['buy_ratio_norm'] = (min15_df['buy_ratio'] - min_buy_ratio) / (max_buy_ratio - min_buy_ratio)
else:
    min15_df['buy_ratio_norm'] = 0.0 # 所有值相同，归一化为 0

In [124]:
bins = np.arange(0, 1.1, 0.1) # [0.0, 0.1, 0.2, ..., 1.0]
labels = [f"{i/10:.1f}-{(i+1)/10:.1f}" for i in range(10)] # ["0.0-0.1", "0.1-0.2", ..., "0.9-1.0"]

In [125]:
min15_df['ratio_group'] = pd.cut(min15_df['buy_ratio_norm'], bins=bins, labels=labels, right=False)

In [128]:
min_zb=min15_df['zb'].min()
max_zb=min15_df['zb'].max()

if max_zb - min_zb != 0:
    min15_df['zb_norm'] = (min15_df['zb'] - min_zb) / (max_zb - min_zb)
else:
    min15_df['zb_norm'] = 0.0 # 所有值相同，归一化为 0

In [136]:
min15_df['zb_group'] = pd.cut(min15_df['zb_norm'], bins=bins, labels=labels, right=False)
min15_df['zf_and_next_sum'] = min15_df['zf'] + min15_df['next_day_zf']

In [137]:
min15_df.groupby('ratio_group').agg({
    'zf': ['count', 
           'mean', 
           'std',
           'sum',
           lambda x:(x>0).sum(),
           lambda x:(x<0).sum()],
    'zf_and_next_sum':
        ['count', 
           'mean', 
           'std',
           'sum',
           lambda x:(x>0).sum(),
           lambda x:(x<0).sum()],
}).reset_index()

C:\Users\cyw\AppData\Local\Temp\ipykernel_29844\3623349965.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  min15_df.groupby('ratio_group').agg({


ratio_group    zf                                                  \
              count      mean       std   sum <lambda_0> <lambda_1>   
0     0.0-0.1   272  0.000221  0.034810  0.06        111        121   
1     0.1-0.2    10  0.038000  0.032249  0.38          8          0   
2     0.2-0.3     4  0.010000  0.008165  0.04          3          0   
3     0.3-0.4     1  0.100000       NaN  0.10          1          0   
4     0.4-0.5     0       NaN       NaN  0.00          0          0   
5     0.5-0.6     1  0.100000       NaN  0.10          1          0   
6     0.6-0.7     1  0.100000       NaN  0.10          1          0   
7     0.7-0.8     0       NaN       NaN  0.00          0          0   
8     0.8-0.9     0       NaN       NaN  0.00          0          0   
9     0.9-1.0     0       NaN       NaN  0.00          0          0   

  zf_and_next_sum                                                  
            count      mean       std   sum <lambda_0> <lambda_1>  
0             271  0.001919  0.050406  0.52        116        117  
1              10  0.055000  0.070435  0.55          8          1  
2               4  0.027500  0.012583  0.11          4          0  
3               1  0.200000       NaN  0.20          1          0  
4               0       NaN       NaN  0.00          0          0  
5               1  0.090000       NaN  0.09          1          0  
6               1  0.200000       NaN  0.20          1          0  
7               0       NaN       NaN  0.00          0          0  
8               0       NaN       NaN  0.00          0          0  
9               0       NaN       NaN  0.00          0          0

C:\Users\cyw\AppData\Local\Temp\ipykernel_29844\2772987570.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  min15_df.groupby('ratio_group').count()


,buy_volume,sell_volume,buy_ratio,all_volume,zb,zf,sjzf,date,next_day_zf,buy_ratio_norm,zb_norm,zb_group
ratio_group,,,,,,,,,,,,
0.0-0.1,272,272,272,272,272,272,272,272,271,272,272,271
0.1-0.2,10,10,10,10,10,10,10,10,10,10,10,10
0.2-0.3,4,4,4,4,4,4,4,4,4,4,4,4
0.3-0.4,1,1,1,1,1,1,1,1,1,1,1,1
0.4-0.5,0,0,0,0,0,0,0,0,0,0,0,0
0.5-0.6,1,1,1,1,1,1,1,1,1,1,1,1
0.6-0.7,1,1,1,1,1,1,1,1,1,1,1,1
0.7-0.8,0,0,0,0,0,0,0,0,0,0,0,0
0.8-0.9,0,0,0,0,0,0,0,0,0,0,0,0


处理完成，前5行数据：
          date                time       code    open    high     low   close  \
48  2025-01-03 2025-01-03 09:35:00  sh.600468  4.5700  4.6100  4.5000  4.5100   
49  2025-01-03 2025-01-03 09:40:00  sh.600468  4.5100  4.5400  4.4800  4.5400   
50  2025-01-03 2025-01-03 09:45:00  sh.600468  4.5200  4.5600  4.5200  4.5500   
96  2025-01-06 2025-01-06 09:35:00  sh.600468  4.3600  4.3700  4.2600  4.3100   
97  2025-01-06 2025-01-06 09:40:00  sh.600468  4.3100  4.3700  4.3000  4.3500   

     volume        amount adjustflag  hour  minute pre_close today_high  \
48  1421000  6478995.0000          3     9      35    4.6000     4.6100   
49  1066300  4805089.0000          3     9      40    4.5100     4.6100   
50   459500  2088137.0000          3     9      45    4.5400     4.6100   
96  1840000  7906074.0000          3     9      35    4.3900     4.4800   
97   773200  3357184.0000          3     9      40    4.3100     4.4800   

    today_open  today_close  yes_close    zf  sjzf

In [102]:
dfs.head(10)

,date,time,code,open,high,low,close,volume,amount,adjustflag,...,pre_close,today_high,today_open,today_close,yes_close,zf,sjzf,buy_ratio,est_buy_vol,est_sell_vol
48,2025-01-03,2025-01-03 09:35:00,sh.600468,4.5700,4.61,4.50,4.51,1421000,6478995.0000,3,...,4.6000,4.6100,4.57,4.39,4.60,-0.05,-0.04,0.09,127890.0,1293110.0
49,2025-01-03,2025-01-03 09:40:00,sh.600468,4.5100,4.54,4.48,4.54,1066300,4805089.0000,3,...,4.5100,4.6100,4.57,4.39,4.60,-0.05,-0.04,1.00,1066300.0,0.0
50,2025-01-03,2025-01-03 09:45:00,sh.600468,4.5200,4.56,4.52,4.55,459500,2088137.0000,3,...,4.5400,4.6100,4.57,4.39,4.60,-0.05,-0.04,0.75,344625.0,114875.0
96,2025-01-06,2025-01-06 09:35:00,sh.600468,4.3600,4.37,4.26,4.31,1840000,7906074.0000,3,...,4.3900,4.4800,4.36,4.46,4.39,0.02,0.02,0.45,828000.0,1012000.0
97,2025-01-06,2025-01-06 09:40:00,sh.600468,4.3100,4.37,4.30,4.35,773200,3357184.0000,3,...,4.3100,4.4800,4.36,4.46,4.39,0.02,0.02,0.71,548972.0,224228.0
98,2025-01-06,2025-01-06 09:45:00,sh.600468,4.3500,4.40,4.34,4.38,684000,2994082.0000,3,...,4.3500,4.4800,4.36,4.46,4.39,0.02,0.02,0.67,458280.0,225720.0
144,2025-01-07,2025-01-07 09:35:00,sh.600468,4.4500,4.49,4.44,4.48,1236200,5526292.0000,3,...,4.4600,4.6400,4.45,4.64,4.46,0.04,0.04,0.80,988960.0,247240.0
145,2025-01-07,2025-01-07 09:40:00,sh.600468,4.4700,4.49,4.44,4.49,644400,2882294.0000,3,...,4.4800,4.6400,4.45,4.64,4.46,0.04,0.04,1.00,644400.0,0.0
146,2025-01-07,2025-01-07 09:45:00,sh.600468,4.4800,4.53,4.47,4.52,1196100,5386043.0000,3,...,4.4900,4.6400,4.45,4.64,4.46,0.04,0.04,0.83,992763.0,203337.0
192,2025-01-08,2025-01-08 09:35:00,sh.600468,4.5800,4.59,4.57,4.59,896800,4107768.0000,3,...,4.6400,4.5900,4.58,4.56,4.64,-0.02,-0.00,1.00,896800.0,0.0


C:\Users\cyw\AppData\Local\Temp\ipykernel_29844\3619589961.py:5: RuntimeWarning: divide by zero encountered in scalar divide
  buy_ratio=buy_volume/sell_volume


In [115]:
min15_df.to_excel(r'C:\Users\cyw\Desktop\jupyternotebook\git-python\GP\当日策列\static\15mincnt.xlsx',index=False)